In [ ]:
"""Train one clean pipeline: ResNet50 + BiLSTM + temporal Attention.

Kaggle: change the dataset roots below and optionally INIT_CHECKPOINT, then Run All.
"""
import copy, random, warnings, hashlib
from pathlib import Path
import cv2, numpy as np, pandas as pd
from PIL import Image
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import resnet50, ResNet50_Weights


In [3]:
!echo "===== DATASET ====="
!find /kaggle/input -type d -name "violence-detection-dataset"

!echo ""
!echo "===== CHECKPOINT ====="
!find /kaggle/input -type f -name "best.pt"

===== DATASET =====
/kaggle/input/datasets/bohong111/datawebcam/A-Dataset-for-Automatic-Violence-Detection-in-Videos-master/violence-detection-dataset

===== CHECKPOINT =====


In [ ]:
from pathlib import Path
import random
import numpy as np
import torch

# =========================
# PATH
# =========================

AIRTLAB_ROOT = Path(
    "/kaggle/input/datasets/bohong111/datawebcam/A-Dataset-for-Automatic-Violence-Detection-in-Videos-master/violence-detection-dataset"
)

RWF_ROOT = Path(
    "/kaggle/input/datasets/vulamnguyen/rwf2000/RWF-2000"
)

RLVS_ROOT = Path(
    "/kaggle/input/datasets/mohamedmustafa/real-life-violence-situations-dataset/Real Life Violence Dataset"
)

INIT_CHECKPOINT = Path(
    "/kaggle/input/models/hoangbao123/modelv1/pytorch/default/1/best.pt"
)

OUT_DIR = Path(
    "/kaggle/working/resnet50_bilstm_attention"
)

MANIFEST_PATH = OUT_DIR / "manifest.csv"

# =========================
# CONFIG
# =========================

# 'combined' gop AIRTLab + RWF-2000 + RLVS | 'airtlab' | 'rwf2000' | 'manifest' (webcam tu quay)
DATA_MODE = "combined"
EVAL_DOMAIN = "external"  # ca 3 dataset deu la 'external', chua phai webcam
CLIP_SPAN_SECONDS = 2.25

SEED = 42
T = 16
SIZE = 224
BATCH = 2
EPOCHS = 20
PATIENCE = 5

CFG = {"seed": SEED}

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

OUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


seed_everything(SEED)


# =========================
# CHECK
# =========================

print("===== KIEM TRA CAU HINH =====")

print("Device:", DEVICE)
print("DATA_MODE:", DATA_MODE)

REQUIRED_ROOTS = {
    "combined": [("AIRTLab", AIRTLAB_ROOT), ("RWF-2000", RWF_ROOT), ("RLVS", RLVS_ROOT)],
    "airtlab": [("AIRTLab", AIRTLAB_ROOT)],
    "rwf2000": [("RWF-2000", RWF_ROOT)],
    "manifest": [],
}[DATA_MODE]

print("\nDataset:")
for name, root in REQUIRED_ROOTS:
    print(f"  {name}: {root} | ton tai: {root.exists()}")
    if not root.exists():
        raise FileNotFoundError(f"Khong tim thay dataset {name}:\n{root}")

print("\nCheckpoint:")
print(INIT_CHECKPOINT)
print("Checkpoint ton tai:", INIT_CHECKPOINT.exists())

if not INIT_CHECKPOINT.exists():
    raise FileNotFoundError(
        f"Khong tim thay model:\n{INIT_CHECKPOINT}"
    )

print("\nOK - Dataset da tim thay.")
print("OK - Model da tim thay.")
print("Co the chay cell tiep theo.")


In [ ]:
def build_airtlab_manifest(data_root, output_csv, seed):
    """Group both cameras of each class/clip ID before stratified splitting."""
    data_root, output_csv = Path(data_root).resolve(), Path(output_csv)
    rows = []
    for class_name, label in [('non-violent', 0), ('violent', 1)]:
        for camera in ('cam1', 'cam2'):
            folder = data_root / class_name / camera
            if not folder.is_dir():
                raise FileNotFoundError(f'Thi\u1ebfu th\u01b0 m\u1ee5c AIRTLab: {folder}')
            videos = sorted(p for p in folder.iterdir()
                            if p.is_file() and p.suffix.lower() in {'.mp4', '.avi', '.mov', '.mkv'})
            if not videos:
                raise ValueError(f'Kh\u00f4ng c\u00f3 video trong {folder}')
            for video in videos:
                if not video.stem.isdecimal():
                    raise ValueError(f'AIRTLab g\u1ed1c d\u00f9ng t\u00ean s\u1ed1, kh\u00f4ng t\u1ef1 \u0111o\u00e1n c\u1eb7p camera: {video.name}')
                cap = cv2.VideoCapture(str(video))
                try:
                    fps = float(cap.get(cv2.CAP_PROP_FPS))
                    count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
                    ok, _ = cap.read()
                    if not cap.isOpened() or not ok or not np.isfinite(fps) or fps <= 0 or count <= 0:
                        raise ValueError(f'Kh\u00f4ng \u0111\u1ecdc \u0111\u01b0\u1ee3c video/FPS: {video}. Ki\u1ec3m tra Input c\u00f3 video th\u1eadt.')
                finally:
                    cap.release()
                rows.append({'path': video.relative_to(data_root).as_posix(), 'label': label,
                             'group_id': f'{class_name}/{int(video.stem)}',
                             'domain': 'external', 'camera': camera, 'last_time': (count - 1) / fps})
    df = pd.DataFrame(rows)
    for group, subset in df.groupby('group_id'):
        if len(subset) != 2 or set(subset.camera) != {'cam1', 'cam2'}:
            raise ValueError(f'C\u1eb7p camera thi\u1ebfu/tr\u00f9ng: {group}')

    # Co clip AIRTLab chi dai 2s: loai CA NHOM neu mot goc khong du 2.25s.
    # Khong keo gian thoi gian hoac lap frame de che viec thieu chuyen dong.
    short_groups = set(df.loc[df.last_time + 1e-6 < CLIP_SPAN_SECONDS, 'group_id'])
    excluded = df[df.group_id.isin(short_groups)].copy()
    excluded['reason'] = 'camera_pair_shorter_than_required_clip_span'
    df = df[~df.group_id.isin(short_groups)].copy()
    rng = np.random.default_rng(seed)
    assignments = {}
    for label in (0, 1):
        groups = sorted(df.loc[df.label == label, 'group_id'].unique())
        if len(groups) < 3:
            raise ValueError(f'L\u1edbp {label} c\u1ea7n \u00edt nh\u1ea5t 3 c\u1eb7p camera h\u1ee3p l\u1ec7 \u0111\u1ec3 chia train/val/test.')
        groups = list(rng.permutation(groups))
        n_val = max(1, round(len(groups) * 0.15))
        n_test = max(1, round(len(groups) * 0.15))
        n_train = len(groups) - n_val - n_test
        for split, selected in [('train', groups[:n_train]),
                                ('val', groups[n_train:n_train + n_val]),
                                ('test', groups[n_train + n_val:])]:
            assignments.update({group: split for group in selected})
    df['split'] = df.group_id.map(assignments)
    output_csv.parent.mkdir(parents=True, exist_ok=True)
    excluded.to_csv(output_csv.parent / 'excluded_airtlab.csv', index=False)
    df[['path', 'label', 'group_id', 'split', 'domain']].to_csv(output_csv, index=False)
    print(f'AIRTLab: {len(df)} video, {df.group_id.nunique()} c\u1eb7p camera; lo\u1ea1i {len(excluded)} video ng\u1eafn/c\u00f9ng c\u1eb7p.')
    print('Manifest:', output_csv)
    return df


def build_rwf_manifest(data_root, output_csv, seed):
    data_root, output_csv = Path(data_root), Path(output_csv)
    rows = []
    for split in ('train', 'val'):
        for folder_name, label in (('NonFight', 0), ('Fight', 1)):
            folder = data_root / split / folder_name
            if not folder.is_dir():
                raise FileNotFoundError(f'Thi\u1ebfu th\u01b0 m\u1ee5c RWF-2000: {folder}')
            for video in sorted(p for p in folder.rglob('*') if p.is_file() and p.suffix.lower() in {'.mp4', '.avi', '.mov', '.mkv'}):
                rel = str(video.relative_to(data_root)).replace('\\', '/')
                # group_id = duong dan tuong doi: RWF khong co cap camera nhu AIRTLab,
                # dung stem thoi se gay 'leak' gia neu 2 file khac thu muc trung ten.
                rows.append({'path': rel, 'label': label, 'group_id': rel, 'source_split': split, 'domain': 'external'})
    df = pd.DataFrame(rows).sample(frac=1, random_state=seed).reset_index(drop=True)
    # Giu split goc cua RWF; val duoc dung lam validation/test noi bo khi can.
    df['split'] = df['source_split'].map({'train': 'train', 'val': 'test'})
    val_rows = df[df.split == 'test'].groupby('label', group_keys=False).apply(lambda x: x.sample(frac=0.5, random_state=seed)).index
    df.loc[val_rows, 'split'] = 'val'
    output_csv.parent.mkdir(parents=True, exist_ok=True)
    df[['path', 'label', 'group_id', 'split', 'domain']].to_csv(output_csv, index=False)
    print(f'RWF-2000: {len(df)} video')
    return df


def build_rlvs_manifest(data_root, output_csv, seed):
    """RLVS khong co cap camera/split goc: moi video la mot nhom, chia nhu AIRTLab."""
    data_root, output_csv = Path(data_root).resolve(), Path(output_csv)
    rows = []
    for folder_name, label in (('NonViolence', 0), ('Violence', 1)):
        folder = data_root / folder_name
        if not folder.is_dir():
            raise FileNotFoundError(f'Thi\u1ebfu th\u01b0 m\u1ee5c RLVS: {folder}')
        videos = sorted(p for p in folder.iterdir()
                        if p.is_file() and p.suffix.lower() in {'.mp4', '.avi', '.mov', '.mkv'})
        if not videos:
            raise ValueError(f'Kh\u00f4ng c\u00f3 video trong {folder}')
        for video in videos:
            rows.append({'path': video.relative_to(data_root).as_posix(), 'label': label,
                         'group_id': f'{folder_name}/{video.stem}', 'domain': 'external'})
    df = pd.DataFrame(rows)
    rng = np.random.default_rng(seed)
    assignments = {}
    for label in (0, 1):
        groups = sorted(df.loc[df.label == label, 'group_id'].unique())
        if len(groups) < 3:
            raise ValueError(f'L\u1edbp {label} c\u1ea7n \u00edt nh\u1ea5t 3 video RLVS \u0111\u1ec3 chia train/val/test.')
        groups = list(rng.permutation(groups))
        n_val = max(1, round(len(groups) * 0.15))
        n_test = max(1, round(len(groups) * 0.15))
        n_train = len(groups) - n_val - n_test
        for split, selected in [('train', groups[:n_train]),
                                ('val', groups[n_train:n_train + n_val]),
                                ('test', groups[n_train + n_val:])]:
            assignments.update({group: split for group in selected})
    df['split'] = df.group_id.map(assignments)
    output_csv.parent.mkdir(parents=True, exist_ok=True)
    df[['path', 'label', 'group_id', 'split', 'domain']].to_csv(output_csv, index=False)
    print(f'RLVS: {len(df)} video, {df.group_id.nunique()} nh\u00f3m.')
    return df


def validate_manifest(csv_path, data_root=None, roots=None):
    csv_path = Path(csv_path)
    if not csv_path.is_file():
        raise FileNotFoundError(f'Kh\u00f4ng t\u00ecm th\u1ea5y manifest: {csv_path}. '
                           'V\u1edbi AIRTLab/RWF/RLVS, ch\u1ea1y tr\u1ecdn cell t\u1ea1o v\u00e0 ki\u1ec3m tra manifest; '
                           'v\u1edbi ch\u1ebf \u0111\u1ed9 manifest, ki\u1ec3m tra \u0111\u01b0\u1eddng d\u1eabn CSV \u0111\u00e3 chu\u1ea9n b\u1ecb.')
    df = pd.read_csv(csv_path, dtype={'path': str, 'group_id': str, 'split': str, 'domain': str, 'source': str})
    required = {'path', 'label', 'group_id', 'split', 'domain'}
    if not required.issubset(df.columns) or df.empty:
        raise ValueError(f'Manifest ph\u1ea3i c\u00f3 d\u1eef li\u1ec7u v\u00e0 c\u00e1c c\u1ed9t {sorted(required)}')
    if roots is not None and 'source' not in df.columns:
        raise ValueError('Manifest g\u1ed9p nhi\u1ec1u dataset c\u1ea7n c\u1ed9t "source" \u0111\u1ec3 bi\u1ebft root g\u1ed1c c\u1ee7a m\u1ed7i video.')
    check_cols = list(required) + (['source'] if roots is not None else [])
    if df[check_cols].isna().any().any():
        raise ValueError('Manifest c\u00f3 \u00f4 tr\u1ed1ng.')
    for col in check_cols:
        if col == 'label':
            continue
        df[col] = df[col].str.strip()
        if df[col].eq('').any():
            raise ValueError(f'C\u1ed9t {col} c\u00f3 gi\u00e1 tr\u1ecb tr\u1ed1ng.')
    if not df.label.isin([0, 1]).all():
        raise ValueError('label ch\u1ec9 nh\u1eadn 0 ho\u1eb7c 1.')
    df['label'] = df.label.astype(int)
    if not df.split.isin(['train', 'val', 'test']).all():
        raise ValueError('split ch\u1ec9 nh\u1eadn train/val/test.')
    if not df.domain.isin(['webcam', 'external']).all():
        raise ValueError('domain ch\u1ec9 nh\u1eadn webcam/external.')
    if roots is not None:
        missing_sources = set(df.source) - set(roots)
        if missing_sources:
            raise ValueError(f'Kh\u00f4ng c\u00f3 root cho source: {sorted(missing_sources)}')
    resolved = []
    for row in df.itertuples():
        if roots is not None:
            base = Path(roots[row.source]).resolve()
        elif data_root is not None:
            base = Path(data_root).resolve()
        else:
            base = csv_path.parent.resolve()
        rel = Path(row.path)
        path = (base / rel).resolve()
        if rel.is_absolute() or not path.is_relative_to(base):
            raise ValueError(f'path ph\u1ea3i t\u01b0\u01a1ng \u0111\u1ed1i v\u00e0 n\u1eb1m trong dataset: {row.path}')
        if not path.is_file():
            raise FileNotFoundError(path)
        resolved.append(str(path))
    df['resolved_path'] = resolved
    if df.resolved_path.duplicated().any():
        raise ValueError('M\u1ed9t file xu\u1ea5t hi\u1ec7n nhi\u1ec1u l\u1ea7n trong manifest.')
    if (df.groupby('group_id').split.nunique() > 1).any():
        raise ValueError('R\u00f2 r\u1ec9 d\u1eef li\u1ec7u: c\u00f9ng group_id xu\u1ea5t hi\u1ec7n \u1edf nhi\u1ec1u split.')
    hashes = []
    for path in tqdm(df.resolved_path, desc='Ki\u1ec3m tra file tr\u00f9ng'):
        digest = hashlib.sha256()
        with open(path, 'rb') as stream:
            for block in iter(lambda: stream.read(1024 * 1024), b''):
                digest.update(block)
        hashes.append(digest.hexdigest())
    df['sha256'] = hashes
    dup_cols = [c for c in ('source', 'split', 'domain', 'path', 'label', 'sha256') if c in df.columns]
    if df.sha256.duplicated(keep=False).any():
        # Video trung noi dung (thuong gap trong RWF-2000/RLVS goc): xu ly tu dong thay vi dung han.
        all_dups = df.loc[df.sha256.duplicated(keep=False)].sort_values('sha256')

        # 1) Trung noi dung nhung nhan mau thuan (vd 1 ban ghi Violence, ban kia NonViolence):
        # khong the tu quyet nhan nao dung -> loai ca nhom, ghi ra file de kiem tra thu cong.
        conflict_mask = all_dups.groupby('sha256').label.transform('nunique') > 1
        conflict_hashes = set(all_dups.loc[conflict_mask, 'sha256'])
        if conflict_hashes:
            conflict_rows = df.loc[df.sha256.isin(conflict_hashes), dup_cols].sort_values('sha256')
            print(f'CANH BAO: {len(conflict_rows)} video trung noi dung nhung nhan mau thuan; loai ca nhom:')
            print(conflict_rows.to_string(index=False))
            conflict_rows.to_csv(csv_path.parent / 'label_conflict_duplicates.csv', index=False)
            df = df[~df.sha256.isin(conflict_hashes)].copy()

        # 2) Trung noi dung, cung nhan, khac split (vd 1 ban o train, 1 ban o val/test):
        # giu dung 1 ban/nhom, uu tien giu ban o test > val > train de bao ve tap danh gia
        # (tranh video train roi vo tinh xuat hien lai o val/test lam sai lech metric).
        split_priority = {'test': 0, 'val': 1, 'train': 2}
        dup_mask = df.sha256.duplicated(keep=False)
        if dup_mask.any():
            dup_group = df.loc[dup_mask].copy()
            dup_group['_prio'] = dup_group.split.map(split_priority)
            keep_idx = dup_group.sort_values(['sha256', '_prio']).groupby('sha256').head(1).index
            drop_idx = dup_group.index.difference(keep_idx)
            dropped = df.loc[drop_idx, dup_cols].sort_values('sha256')
            print(f'Loai {len(drop_idx)} video trung noi dung (giu 1 ban/nhom, uu tien test > val > train):')
            print(dropped.to_string(index=False))
            dropped.to_csv(csv_path.parent / 'duplicate_videos_removed.csv', index=False)
            df = df.drop(index=drop_idx).copy()

        df = df.reset_index(drop=True)
    for split in ('train', 'val', 'test'):
        target = df[(df.split == split) & (df.domain == EVAL_DOMAIN)]
        if set(target.label) != {0, 1}:
            raise ValueError(f'{split} {EVAL_DOMAIN} c\u1ea7n \u0111\u1ee7 2 l\u1edbp \u0111\u1ec3 train/\u0111\u00e1nh gi\u00e1 \u0111\u00fang m\u1ee5c ti\u00eau.')
        if target.group_id.nunique() < 3:
            warnings.warn(f'{split} {EVAL_DOMAIN} c\u00f3 d\u01b0\u1edbi 3 nh\u00f3m; s\u1ed1 li\u1ec7u c\u00f3 th\u1ec3 dao \u0111\u1ed9ng m\u1ea1nh.')
    return df


def prepare_manifest():
    # Cung cell dinh nghia builder va validator: khong phu thuoc cell tao CSV rieng.
    # Giu manifest da ton tai de khong tu thay doi split khi chay lai cell.
    if DATA_MODE == 'combined' and not Path(MANIFEST_PATH).is_file():
        print('Ch\u01b0a c\u00f3 manifest; \u0111ang g\u1ed9p AIRTLab + RWF-2000 + RLVS...')
        parts = []
        for source, builder, root in (
            ('airtlab', build_airtlab_manifest, AIRTLAB_ROOT),
            ('rwf2000', build_rwf_manifest, RWF_ROOT),
            ('rlvs', build_rlvs_manifest, RLVS_ROOT),
        ):
            part = builder(root, OUT_DIR / f'manifest_{source}.csv', CFG['seed']).copy()
            part['source'] = source
            part['group_id'] = source + '/' + part['group_id'].astype(str)
            parts.append(part)
        combined = pd.concat(parts, ignore_index=True)
        Path(MANIFEST_PATH).parent.mkdir(parents=True, exist_ok=True)
        combined[['path', 'label', 'group_id', 'split', 'domain', 'source']].to_csv(MANIFEST_PATH, index=False)
    if DATA_MODE == 'rwf2000' and not Path(MANIFEST_PATH).is_file():
        print('\u0110ang t\u1ea1o manifest t\u1eeb RWF-2000:', RWF_ROOT)
        build_rwf_manifest(RWF_ROOT, MANIFEST_PATH, CFG['seed'])
    if DATA_MODE == 'airtlab' and not Path(MANIFEST_PATH).is_file():
        print('Ch\u01b0a c\u00f3 manifest; \u0111ang t\u1ea1o t\u1eeb AIRTLab:', AIRTLAB_ROOT)
        build_airtlab_manifest(AIRTLAB_ROOT, MANIFEST_PATH, CFG['seed'])
    if DATA_MODE == 'combined':
        print('\u0110\u00e1nh gi\u00e1 tr\u00ean AIRTLab + RWF-2000 + RLVS g\u1ed9p (external), ch\u01b0a ph\u1ea3i \u0111\u1ed9 ch\u00ednh x\u00e1c webcam th\u1ef1c t\u1ebf.')
    elif DATA_MODE == 'rwf2000':
        print('\u0110\u00e1nh gi\u00e1 tr\u00ean RWF-2000 (external), ch\u01b0a ph\u1ea3i \u0111\u1ed9 ch\u00ednh x\u00e1c webcam th\u1ef1c t\u1ebf.')
    elif DATA_MODE == 'airtlab':
        print('\u0110\u00e1nh gi\u00e1 tr\u00ean AIRTLab (external), ch\u01b0a ph\u1ea3i \u0111\u1ed9 ch\u00ednh x\u00e1c webcam th\u1ef1c t\u1ebf.')
    roots = {'airtlab': AIRTLAB_ROOT, 'rwf2000': RWF_ROOT, 'rlvs': RLVS_ROOT} if DATA_MODE == 'combined' else None
    data_root = AIRTLAB_ROOT if DATA_MODE == 'airtlab' else RWF_ROOT if DATA_MODE == 'rwf2000' else None
    return validate_manifest(MANIFEST_PATH, data_root=data_root, roots=roots)


manifest = prepare_manifest()
print(manifest.groupby(['split', 'domain', 'label']).size().rename('videos').to_string())
manifest.to_csv(OUT_DIR / 'manifest_resolved.csv', index=False)


## 3. Lấy frame theo thời gian và kiểm tra ảnh đầu vào

Train: cửa sổ ngẫu nhiên trong video đã gán nhãn, 16 frame cách nhau 150 ms.
Validation/test: duyệt cửa sổ cố định bước 1,1 giây, thêm cửa sổ cuối video.
Đánh giá **theo cửa sổ**, không lấy trung bình toàn video để che mất hành động ngắn.
Video ngắn hơn cửa sổ hoặc frame lỗi sẽ báo lỗi, không tạo ảnh đen thay thế.

Mô phỏng đường truyền webcam bằng resize cạnh dài tối đa 384 và JPEG quality 70
(xấp xỉ browser JPEG 0.7; bộ mã hóa trình duyệt có thể khác).
Sau đó dùng Resize(size + 32), CenterCrop và chuẩn hóa ImageNet giống web.
Augmentation train dùng cùng thông số cho cả clip để giữ chuyển động nhất quán.

In [ ]:
def collect():
    # Nguon du lieu: manifest da gop AIRTLab + RWF-2000 + RLVS (hoac 1 dataset tuy DATA_MODE),
    # duoc tao/kiem tra o cell truoc (bien `manifest`, cot `resolved_path`).
    print("===== DATASET (manifest) =====")
    print(manifest.groupby(["split", "label"]).size().to_string())

    def split_xy(name):
        subset = manifest[manifest.split == name]
        return subset.resolved_path.tolist(), subset.label.astype(int).tolist()

    train_p, train_y = split_xy("train")
    val_p, val_y = split_xy("val")
    test_p, test_y = split_xy("test")

    print("\n===== SPLIT =====")
    print("Train:", len(train_p))
    print("Val  :", len(val_p))
    print("Test :", len(test_p))

    print("\nTrain:")
    print("  Non-violent:", train_y.count(0))
    print("  Violent    :", train_y.count(1))

    print("\nVal:")
    print("  Non-violent:", val_y.count(0))
    print("  Violent    :", val_y.count(1))

    print("\nTest:")
    print("  Non-violent:", test_y.count(0))
    print("  Violent    :", test_y.count(1))

    return (
        train_p,
        train_y,
        val_p,
        val_y,
        test_p,
        test_y
    )


## 4. CNN + BiLSTM + Attention

Các lớp bên dưới được đồng bộ từ source hiện tại để giữ tên và kích thước trọng số.
Mặc định ResNet50; có thể chọn ResNet18 khi train mới để giảm chi phí CPU localhost.
Nếu nạp checkpoint cũ, kiến trúc được lấy từ checkpoint và kiểm tra bằng `strict=True`.

In [ ]:
MEAN, STD = (0.485, .456, .406), (.229, .224, .225)
train_tf = transforms.Compose([
    transforms.Resize((SIZE+32, SIZE+32)), transforms.RandomCrop(SIZE),
    transforms.RandomHorizontalFlip(), transforms.ToTensor(),
    transforms.Normalize(MEAN, STD)])
eval_tf = transforms.Compose([
    transforms.Resize((SIZE, SIZE)), transforms.ToTensor(),
    transforms.Normalize(MEAN, STD)])



In [ ]:
def read_clip(path, train):
    cap = cv2.VideoCapture(str(path)); n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if not cap.isOpened() or n <= 0:
        cap.release(); raise RuntimeError(f"Không đọc được video: {path}")
    # Random contiguous clip only for train; fixed evenly-spaced clip for eval.
    start = random.randint(0, max(0, n-T)) if train else max(0, (n-T)//2)
    frames = []
    for idx in range(start, min(start+T, n)):
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx); ok, bgr = cap.read()
        if ok: frames.append(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB))
    cap.release()
    if not frames: raise RuntimeError(f"Không decode được video: {path}")
    while len(frames) < T: frames.append(frames[-1])
    tf = train_tf if train else eval_tf
    return torch.stack([tf(Image.fromarray(f)) for f in frames[:T]])

class VideoDataset(Dataset):
    def __init__(self, paths, labels, train=False):
        self.paths, self.labels, self.train = paths, labels, train
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        return read_clip(self.paths[i], self.train), torch.tensor(self.labels[i], dtype=torch.long)



## 5. Huấn luyện và chọn checkpoint

Warmup LSTM/attention, sau đó fine-tune tầng cuối CNN bằng learning rate nhỏ.
AMP giảm bộ nhớ GPU, gradient clipping hạn chế gradient lớn.
Chọn checkpoint bằng macro-F1 validation tại ngưỡng 0.5 trên `EVAL_DOMAIN`:
`external` cho AIRTLab, `webcam` cho chế độ manifest webcam. Không dùng test để chọn model.

In [ ]:
class TemporalAttention(nn.Module):
    def __init__(self, d):
        super().__init__(); self.score = nn.Sequential(nn.Linear(d,d//2), nn.Tanh(), nn.Linear(d//2,1))
    def forward(self, h):
        a = torch.softmax(self.score(h).squeeze(-1), dim=1)
        return torch.bmm(a.unsqueeze(1), h).squeeze(1), a

class ResNet50BiLSTMAttention(nn.Module):
    def __init__(self):
        super().__init__(); net = resnet50(weights=ResNet50_Weights.DEFAULT)
        self.cnn = nn.Sequential(*list(net.children())[:-1])
        self.lstm = nn.LSTM(2048, 512, 2, batch_first=True, bidirectional=True, dropout=.5)
        self.attention = TemporalAttention(1024)
        self.head = nn.Sequential(nn.Dropout(.5), nn.Linear(1024, 2))
    def forward(self, x):
        b,t,c,h,w = x.shape; z = self.cnn(x.reshape(b*t,c,h,w)).flatten(1)
        z,_ = self.lstm(z.view(b,t,2048)); context, alpha = self.attention(z)
        return self.head(context), alpha



## 6. Chọn ngưỡng từ validation, rồi đánh giá test

Tối đa macro-F1 trên validation của `EVAL_DOMAIN`, báo cáo precision, recall và tỷ lệ báo nhầm.
Ngưỡng AIRTLab chưa được hiệu chỉnh trên webcam thật. Không dùng test để chỉnh ngưỡng.
Các cửa sổ chồng lấn không phải các mẫu độc lập. Đây chưa phải đo cảnh báo/phút hoặc độ trễ camera.

In [ ]:
def load_old(model, path):
    if not path: return model
    ckpt = torch.load(path, map_location="cpu", weights_only=False)
    state = ckpt.get("model", ckpt); mapped = {}
    # Supports project checkpoints (spatial/temporal) and this notebook format.
    for k,v in state.items():
        k = k.replace("spatial.features.", "cnn.").replace("temporal.lstm.", "lstm.")
        k = k.replace("temporal.attn.", "attention.").replace("temporal.classifier.", "head.")
        mapped[k] = v
    missing, unexpected = model.load_state_dict(mapped, strict=False)
    print("Loaded checkpoint:", path, "missing:", len(missing), "unexpected:", len(unexpected))
    return model



In [ ]:
def run_epoch(model, loader, criterion, optimizer=None):
    train = optimizer is not None; model.train(train); ys=[]; ps=[]; losses=[]
    with (torch.enable_grad() if train else torch.no_grad()):
        for x,y in tqdm(loader, leave=False):
            x,y = x.to(DEVICE), y.to(DEVICE)
            if train: optimizer.zero_grad(set_to_none=True)
            logits,_ = model(x); loss = criterion(logits,y)
            if train: loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0); optimizer.step()
            losses.append(loss.item()); ys += y.cpu().tolist(); ps += logits.argmax(1).cpu().tolist()
    return {"loss": float(np.mean(losses)), "accuracy": accuracy_score(ys,ps),
            "f1": f1_score(ys,ps,average="macro",zero_division=0)}



In [ ]:
train_p, train_y, val_p, val_y, test_p, test_y = collect()
loader = lambda p,y,tr=False: DataLoader(VideoDataset(p,y,tr), batch_size=BATCH, shuffle=tr,
                                           num_workers=0, pin_memory=DEVICE.type=="cuda")
train_loader, val_loader, test_loader = loader(train_p,train_y,True), loader(val_p,val_y), loader(test_p,test_y)
model = load_old(ResNet50BiLSTMAttention().to(DEVICE), INIT_CHECKPOINT)
criterion = nn.CrossEntropyLoss(); optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5, weight_decay=1e-4)
best, stale, best_path = -1.0, 0, OUT_DIR / "best.pt"
print("Device:", DEVICE, "train/val/test:", len(train_p), len(val_p), len(test_p))
for epoch in range(1,EPOCHS+1):
    tr = run_epoch(model, train_loader, criterion, optimizer); va = run_epoch(model,val_loader,criterion)
    print(f"epoch {epoch:02d} train={tr} val={va}")
    if va["f1"] > best:
        best, stale = va["f1"], 0
        torch.save({"model":model.state_dict(), "cfg":{"data":{"classes":["non_violence","violence"],"num_frames":T,"image_size":SIZE},
            "model":{"backbone":"resnet50","pretrained":False,"freeze_backbone":False,"lstm_hidden":512,"lstm_layers":2,"bidirectional":True,"dropout":.5,"use_attention":True,"num_classes":2},
            "infer":{"window_frames":T,"stride":8,"threshold":.6}}, "val":va}, best_path)
        print("saved", best_path)
    else:
        stale += 1
        if stale >= PATIENCE: break
print("TEST:", run_epoch(model,test_loader,criterion)); print("BEST:", best_path)


## 7. Đưa checkpoint về localhost

1. Tải `webcam_cnn_lstm/best.pt` và báo cáo từ Output Kaggle. Giữ một bản checkpoint cũ để so sánh.
2. Chép checkpoint mới vào `checkpoints/best.pt` trong project và **restart** web:

   ```powershell
   python -m uvicorn src.api:app --host 127.0.0.1 --port 8000
   ```

3. Xem `/health`, sau đó mở `/live`. Backend đọc cấu hình và ngưỡng từ checkpoint mới.
   Không cần sửa ngưỡng trong `default.yaml`. Ngưỡng được chọn trên domain ghi trong checkpoint; AIRTLab chưa thay thế việc hiệu chỉnh webcam thật.
4. Frontend hiện lấy 16 frame cách khoảng 150 ms, giảm cạnh dài còn 384 và JPEG quality 0.7;
   notebook mô phỏng thông số này. Chỉ đánh giá sau khi buffer đủ 16 frame (khoảng 2,4 giây).
   Frontend cũ cho phép request từ 8 frame; những dự đoán khởi động đó chưa khớp luồng train.
   Thời gian lấy frame trình duyệt có thể trễ; cần đo lại trên máy sử dụng thật.
5. Thử các buổi quay mới, ghi nhận báo nhầm và bỏ sót. Không dùng lại test này để train rồi tiếp tục
   báo cáo nó như test độc lập. Muốn fine-tune tiếp phải giữ một test khác chưa sử dụng.

**Phạm vi tương thích:** checkpoint nạp được vào backend hiện tại; kiến trúc, RGB, resize/crop và
chuẩn hóa giống source. Notebook này thay đổi train/evaluation, không thay source web.
`src.infer` ở đường **upload** vẫn dùng cửa sổ frame liên tiếp; công cụ đánh giá local cũ đã được gỡ.
Đường upload chưa tái hiện cách lấy mẫu theo thời gian của notebook. Không so trực tiếp các số liệu đó
với `metrics.json`; dùng notebook để đánh giá video webcam theo đúng protocol này.

Khung người HOG trên web không phải định vị người đang bạo lực. Phần trăm là xác suất cho cả clip.
Nếu kết quả vẫn kém, xem các cửa sổ dự đoán sai trong CSV, kiểm tra crop/FPS/nhãn và bổ sung dữ liệu
webcam tương ứng trước khi tăng kích thước mô hình.